# NiyamTrace-X — External Dataset Validation V3.1
## Script-free, Colab-safe, Scientific Reports experiment

This notebook is the hardened replacement for the earlier MASSIVE/MTOP notebook.

It deliberately avoids the two failure classes already observed:

1. **No scikit-learn / NumPy ABI replacement**
2. **No Hugging Face dataset loading scripts / `trust_remote_code`**

### External datasets

**MASSIVE**
- Source: official Amazon MASSIVE release (`amazon-massive-dataset-1.0.tar.gz`)
- Natural multilingual utterances
- Intent + slot annotations
- Languages used by default: English, Hindi, Telugu

**MTOP**
- Source: script-free JSONL files from `mteb/mtop_intent`
- Independent multilingual intent benchmark
- Languages used by default: English, Hindi

**MLCL**
- Optional multilingual tool-calling extension
- If automatic artifact retrieval is unavailable, upload the repository/data ZIP
- The notebook continues even when MLCL is unavailable

### Paper claim discipline

This notebook measures **external semantic and tool-interface generalization**.

It does **not** replace the frozen NiyamTrace-X authorization benchmark and does
not by itself establish end-to-end authorization safety.

In [ ]:
# CELL 1 — SAFE ENVIRONMENT CHECK
# No NumPy/Pandas/SciPy/sklearn replacement is performed.

import sys, os, subprocess, importlib.util

print("Python:", sys.version)

# Install only packages that are missing. Do not replace scientific-core packages.
REQUIRED = {
    "requests": "requests>=2.31",
    "tqdm": "tqdm>=4.66",
    "matplotlib": "matplotlib>=3.8",
    "transformers": "transformers>=4.48",
    "accelerate": "accelerate>=1.2",
    "huggingface_hub": "huggingface_hub>=0.27",
}

missing = [
    req for module, req in REQUIRED.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing missing packages only:")
    for x in missing:
        print(" -", x)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        *missing,
    ])
else:
    print("✅ All required packages are already present.")

print("✅ Environment setup complete.")

In [ ]:
# CELL 2 — IMPORTS + ABI SANITY CHECK + CONFIGURATION

from pathlib import Path
from collections import defaultdict
from typing import Any, Dict, List, Tuple
import os, re, json, math, time, random, hashlib, zipfile, shutil, subprocess
import tarfile, io

import requests
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

# Catch a damaged runtime before any long work.
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)

a = np.asarray([1.0, 2.0, 3.0])
assert np.isclose(a.mean(), 2.0)
assert int(pd.DataFrame({"x":[1,2,3]})["x"].sum()) == 6
print("✅ NumPy/Pandas sanity check passed.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = Path("/content/NTX_EXTERNAL_DATASET_VALIDATION_V3")
DATA = BASE / "data"
RAW = BASE / "raw_predictions"
RESULTS = BASE / "results"
FIGURES = BASE / "figures"
EXPORT = BASE / "export"
CACHE = BASE / "cache"

for p in [BASE, DATA, RAW, RESULTS, FIGURES, EXPORT, CACHE]:
    p.mkdir(parents=True, exist_ok=True)

MODE = os.getenv("NTX_EXT_MODE", "PAPER").upper()
assert MODE in {"SMOKE", "PAPER", "FULL"}

LIMITS = {
    "SMOKE": {"massive_per_lang":20, "mtop_per_lang":20, "mlcl_per_lang":20, "bootstrap":500},
    "PAPER": {"massive_per_lang":150, "mtop_per_lang":150, "mlcl_per_lang":100, "bootstrap":5000},
    "FULL":  {"massive_per_lang":500, "mtop_per_lang":500, "mlcl_per_lang":None, "bootstrap":10000},
}[MODE]

MASSIVE_LANGS = ["en-US", "hi-IN", "te-IN"]
MTOP_LANGS = ["en", "hi"]

LOCAL_MODEL = os.getenv("NTX_LOCAL_MODEL", "Qwen/Qwen2.5-3B-Instruct")
MAX_NEW_TOKENS = 320

HF_TOKEN = os.getenv("HF_TOKEN", "").strip() or None

print("MODE:", MODE)
print("LIMITS:", LIMITS)
print("MODEL:", LOCAL_MODEL)

### If an earlier notebook modified NumPy

If you previously ran a notebook that upgraded NumPy and the current session
already produced `_blas_supports_fpe`, restart the Colab runtime once before
continuing:

**Runtime → Restart session**

This V3 notebook itself does not replace NumPy/Pandas/scikit-learn.

In [ ]:
# CELL 3 — NETWORK HELPERS WITH RETRIES

def http_get(url, timeout=60, stream=False, headers=None, attempts=4):
    last = None
    for attempt in range(1, attempts+1):
        try:
            r = requests.get(
                url,
                timeout=timeout,
                stream=stream,
                headers=headers or {},
                allow_redirects=True,
            )
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            wait = min(2**attempt, 12)
            print(f"HTTP attempt {attempt}/{attempts} failed:", repr(e))
            if attempt < attempts:
                time.sleep(wait)
    raise RuntimeError(f"Download failed after {attempts} attempts: {url}\n{last}")

def download_stream(url, dest: Path, chunk=1024*1024):
    if dest.exists() and dest.stat().st_size > 0:
        print("Using cached:", dest)
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    if tmp.exists():
        tmp.unlink()
    r = http_get(url, timeout=120, stream=True)
    total = int(r.headers.get("content-length", 0))
    with tmp.open("wb") as f:
        with tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as bar:
            for block in r.iter_content(chunk_size=chunk):
                if block:
                    f.write(block)
                    bar.update(len(block))
    tmp.replace(dest)
    return dest

print("✅ Network helpers ready.")

In [ ]:
# CELL 4 — DOWNLOAD MASSIVE FROM THE OFFICIAL AMAZON RELEASE
#
# This completely bypasses the deprecated Hugging Face dataset loading script.

MASSIVE_URL = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.0.tar.gz"
MASSIVE_TAR = CACHE / "amazon-massive-dataset-1.0.tar.gz"
MASSIVE_EXTRACT = CACHE / "massive_1_0"

download_stream(MASSIVE_URL, MASSIVE_TAR)

if not MASSIVE_EXTRACT.exists():
    MASSIVE_EXTRACT.mkdir(parents=True, exist_ok=True)
    print("Extracting MASSIVE...")
    with tarfile.open(MASSIVE_TAR, "r:gz") as tf:
        # Security-safe extraction.
        root = MASSIVE_EXTRACT.resolve()
        for member in tf.getmembers():
            target = (MASSIVE_EXTRACT / member.name).resolve()
            if not str(target).startswith(str(root)):
                raise RuntimeError("Unsafe archive path: " + member.name)
        tf.extractall(MASSIVE_EXTRACT)
    print("✅ MASSIVE extracted.")
else:
    print("Using extracted MASSIVE cache.")

# Locate official locale JSONL files.
massive_files = {}
for lang in MASSIVE_LANGS:
    matches = list(MASSIVE_EXTRACT.rglob(f"{lang}.jsonl"))
    if not matches:
        raise FileNotFoundError(f"MASSIVE locale not found after official extraction: {lang}")
    massive_files[lang] = matches[0]
    print(lang, "->", matches[0])

print("✅ MASSIVE official files found for all requested locales.")

In [ ]:
# CELL 5 — PARSE MASSIVE OFFICIAL JSONL + SAMPLE TEST PARTITION

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

massive_rows = []

for lang, path in massive_files.items():
    rows = read_jsonl(path)

    # Official MASSIVE JSONL stores all partitions in each locale file.
    test_rows = [r for r in rows if str(r.get("partition","")).lower() == "test"]
    if not test_rows:
        raise RuntimeError(f"No test partition rows found in {path}")

    n = min(LIMITS["massive_per_lang"], len(test_rows))
    rng = np.random.default_rng(SEED)
    idx = sorted(rng.choice(len(test_rows), size=n, replace=False).tolist())

    for i in idx:
        r = test_rows[i]
        massive_rows.append({
            "dataset": "MASSIVE",
            "language": lang,
            "id": str(r.get("id","")),
            "text": str(r.get("utt","")),
            "gold_intent": str(r.get("intent","")),
            "gold_scenario": str(r.get("scenario","")),
            "gold_slots_raw": str(r.get("annot_utt","")),
        })

massive_df = pd.DataFrame(massive_rows)
assert len(massive_df) == len(MASSIVE_LANGS) * LIMITS["massive_per_lang"]
assert massive_df["text"].str.len().gt(0).all()
assert massive_df["gold_intent"].str.len().gt(0).all()

massive_df.to_csv(DATA / "massive_eval.csv", index=False)

print("MASSIVE rows:", len(massive_df))
print(massive_df.groupby("language").size())
display(massive_df.head())

In [ ]:
# CELL 6 — DOWNLOAD MTOP INTENT AS STANDARD JSONL FILES
#
# We use mteb/mtop_intent because it contains ordinary JSONL/Parquet assets and
# therefore does not depend on the retired MTOP loading script.
#
# MTOP in this notebook is an intent-generalization experiment only.
# MASSIVE supplies the external slot-grounding experiment.

mtop_rows = []

for lang in MTOP_LANGS:
    filename = f"{lang}/test.jsonl"
    try:
        local_path = hf_hub_download(
            repo_id="mteb/mtop_intent",
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )
    except Exception as e:
        raise RuntimeError(
            f"Could not download script-free MTOP JSONL {filename}: {e}"
        )

    rows = read_jsonl(local_path)
    if not rows:
        raise RuntimeError(f"Empty MTOP file: {local_path}")

    n = min(LIMITS["mtop_per_lang"], len(rows))
    rng = np.random.default_rng(SEED)
    idx = sorted(rng.choice(len(rows), size=n, replace=False).tolist())

    for i in idx:
        r = rows[i]
        mtop_rows.append({
            "dataset": "MTOP",
            "language": lang,
            "id": str(r.get("id", i)),
            "text": str(r.get("text","")),
            "gold_intent": str(r.get("label_text", r.get("label",""))),
        })

mtop_df = pd.DataFrame(mtop_rows)
assert len(mtop_df) == len(MTOP_LANGS) * LIMITS["mtop_per_lang"]
assert mtop_df["text"].str.len().gt(0).all()
assert mtop_df["gold_intent"].str.len().gt(0).all()

mtop_df.to_csv(DATA / "mtop_intent_eval.csv", index=False)

print("MTOP rows:", len(mtop_df))
print(mtop_df.groupby("language").size())
display(mtop_df.head())

In [ ]:
# CELL 7 — SELF-TEST DATA SCHEMAS BEFORE MODEL LOADING

# MASSIVE sample expected fields
required_massive = {"language","text","gold_intent","gold_slots_raw"}
required_mtop = {"language","text","gold_intent"}

assert required_massive.issubset(massive_df.columns)
assert required_mtop.issubset(mtop_df.columns)

# No accidental duplicate row keys within each selected set.
assert not massive_df[["language","id"]].duplicated().any()
assert not mtop_df[["language","id"]].duplicated().any()

print("MASSIVE intents:", massive_df["gold_intent"].nunique())
print("MTOP intents:", mtop_df["gold_intent"].nunique())
print("✅ Dataset schema self-test passed.")

In [ ]:
# CELL 8 — SLOT PARSER + METRICS (NO SCIKIT-LEARN)

def normalize_text(x):
    x = "" if x is None else str(x)
    x = x.strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"[\"'`]", "", x)
    return x

def parse_massive_slots(annot_utt):
    if not isinstance(annot_utt, str):
        return []
    pairs = []

    # Official MASSIVE examples use [slot : value]
    for m in re.finditer(r"\[\s*([^:\]]+?)\s*:\s*([^\]]+?)\s*\]", annot_utt):
        pairs.append((m.group(1).strip(), m.group(2).strip()))

    if not pairs:
        # Defensive fallback for [SL:label value]
        for m in re.finditer(r"\[\s*(?:SL:)?([A-Za-z0-9_]+)\s+([^\]]+?)\s*\]", annot_utt):
            pairs.append((m.group(1).strip(), m.group(2).strip()))
    return pairs

def slot_set(slots):
    return {(normalize_text(k), normalize_text(v)) for k,v in slots}

def label_set(slots):
    return {normalize_text(k) for k,_ in slots}

def prf(gold, pred):
    gold, pred = set(gold), set(pred)
    tp = len(gold & pred)
    fp = len(pred - gold)
    fn = len(gold - pred)
    precision = tp/(tp+fp) if tp+fp else (1.0 if not gold else 0.0)
    recall = tp/(tp+fn) if tp+fn else 1.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    return precision, recall, f1

def bootstrap_ci(values, n_boot=None, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if not len(values):
        return np.nan, np.nan, np.nan
    n_boot = n_boot or LIMITS["bootstrap"]
    rng = np.random.default_rng(SEED)
    reps = np.empty(n_boot)
    for i in range(n_boot):
        reps[i] = rng.choice(values, size=len(values), replace=True).mean()
    return (
        float(values.mean()),
        float(np.quantile(reps, alpha/2)),
        float(np.quantile(reps, 1-alpha/2)),
    )

# Unit test the parser on a MASSIVE-style annotation.
test_slots = parse_massive_slots("set alarm for [time : five am] [date : tomorrow]")
assert ("time","five am") in test_slots
assert ("date","tomorrow") in test_slots

print("✅ Metric and slot-parser self-tests passed.")

In [ ]:
# CELL 9 — LOAD LOCAL MODEL IN FP16
#
# No bitsandbytes is used. Qwen2.5-3B in FP16 fits comfortably on a T4.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab choose Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("Loading:", LOCAL_MODEL)

tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL,
    token=HF_TOKEN,
    trust_remote_code=False,
)

model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=False,
)
model.eval()

print("✅ Model loaded in FP16.")

In [ ]:
# CELL 10 — ROBUST MODEL CALL + JSON EXTRACTION

def extract_json(text):
    if not isinstance(text, str):
        return None
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    if start < 0:
        return None

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    chunk = text[start:i+1]
                    try:
                        return json.loads(chunk)
                    except Exception:
                        return None
    return None

def call_llm(system, user, max_new_tokens=MAX_NEW_TOKENS):
    messages = [
        {"role":"system","content":system},
        {"role":"user","content":user},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    enc = tokenizer(prompt, return_tensors="pt")
    enc = {k:v.to(model.device) for k,v in enc.items()}

    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out[0, enc["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# Quick inference self-test before the long experiments.
test = call_llm(
    "Return only JSON.",
    'Return {"ok": true}.',
    max_new_tokens=32,
)
print("Model self-test:", test[:200])
assert isinstance(test, str) and len(test) > 0
print("✅ Model inference self-test passed.")

In [ ]:
# CELL 11 — MASSIVE INTENT + SLOT EVALUATION

MASSIVE_SYSTEM = """You are evaluating multilingual task-oriented language understanding.
Return ONLY valid JSON:
{
  "intent": "one exact allowed intent label",
  "slots": [{"label":"slot_label","value":"surface value from the utterance"}]
}
Rules:
- Select only an intent from the supplied list.
- Never invent a slot value unsupported by the utterance.
- Slot values should preserve the user's surface meaning.
- Use only slot labels from the supplied list.
- Do not add explanations.
"""

all_massive_intents = sorted({
    r["intent"]
    for path in massive_files.values()
    for r in read_jsonl(path)
    if r.get("intent")
})

all_massive_slot_labels = sorted({
    label
    for path in massive_files.values()
    for r in read_jsonl(path)
    for label,_ in parse_massive_slots(str(r.get("annot_utt","")))
})

print("Allowed MASSIVE intents:", len(all_massive_intents))
print("Allowed MASSIVE slot labels:", len(all_massive_slot_labels))

pred_path = RAW / "massive_predictions.jsonl"
done = {}
if pred_path.exists():
    for line in pred_path.read_text(encoding="utf-8").splitlines():
        try:
            obj=json.loads(line); done[obj["row_key"]]=obj
        except Exception:
            pass

with pred_path.open("a", encoding="utf-8") as fout:
    for idx,row in tqdm(massive_df.iterrows(), total=len(massive_df), desc="MASSIVE"):
        row_key = f"{row['language']}::{row['id']}"
        if row_key in done:
            continue

        user = f"""Language: {row['language']}
Utterance: {row['text']}

Allowed intents:
{json.dumps(all_massive_intents, ensure_ascii=False)}

Allowed slot labels:
{json.dumps(all_massive_slot_labels, ensure_ascii=False)}
"""
        raw = call_llm(MASSIVE_SYSTEM, user)
        obj = extract_json(raw) or {}

        pred_slots_raw = obj.get("slots", [])
        if not isinstance(pred_slots_raw, list):
            pred_slots_raw=[]

        pred_slots=[]
        for s in pred_slots_raw:
            if isinstance(s, dict):
                pred_slots.append((str(s.get("label","")), str(s.get("value",""))))

        gold_slots = parse_massive_slots(row["gold_slots_raw"])

        pp,pr,pf = prf(slot_set(gold_slots), slot_set(pred_slots))
        lp,lr,lf = prf(label_set(gold_slots), label_set(pred_slots))

        grounded = [
            normalize_text(v) in normalize_text(row["text"])
            for _,v in pred_slots if normalize_text(v)
        ]

        rec = {
            "row_key":row_key,
            "dataset":"MASSIVE",
            "language":row["language"],
            "id":row["id"],
            "text":row["text"],
            "gold_intent":row["gold_intent"],
            "pred_intent":str(obj.get("intent","")),
            "intent_correct":normalize_text(obj.get("intent","")) == normalize_text(row["gold_intent"]),
            "gold_slots":gold_slots,
            "pred_slots":pred_slots,
            "slot_pair_precision":pp,
            "slot_pair_recall":pr,
            "slot_pair_f1":pf,
            "slot_label_precision":lp,
            "slot_label_recall":lr,
            "slot_label_f1":lf,
            "surface_grounded":all(grounded) if grounded else True,
            "hallucinated_slot_value_rate":1-(sum(grounded)/len(grounded)) if grounded else 0.0,
            "frame_exact":(
                normalize_text(obj.get("intent","")) == normalize_text(row["gold_intent"])
                and slot_set(gold_slots) == slot_set(pred_slots)
            ),
            "raw_output":raw,
        }
        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        fout.flush()

print("✅ MASSIVE evaluation complete.")

In [ ]:
# CELL 12 — MTOP INTENT EVALUATION

MTOP_SYSTEM = """You are evaluating multilingual intent classification.
Return ONLY valid JSON:
{"intent":"one exact allowed intent label"}
Use only the supplied labels. Do not add explanation.
"""

all_mtop_intents = sorted(mtop_df["gold_intent"].astype(str).unique())

pred_path = RAW / "mtop_predictions.jsonl"
done = {}
if pred_path.exists():
    for line in pred_path.read_text(encoding="utf-8").splitlines():
        try:
            obj=json.loads(line); done[obj["row_key"]]=obj
        except Exception:
            pass

with pred_path.open("a", encoding="utf-8") as fout:
    for _,row in tqdm(mtop_df.iterrows(), total=len(mtop_df), desc="MTOP"):
        row_key = f"{row['language']}::{row['id']}"
        if row_key in done:
            continue

        user = f"""Language: {row['language']}
Utterance: {row['text']}

Allowed intents:
{json.dumps(all_mtop_intents, ensure_ascii=False)}
"""
        raw = call_llm(MTOP_SYSTEM, user, max_new_tokens=96)
        obj = extract_json(raw) or {}

        rec = {
            "row_key":row_key,
            "dataset":"MTOP",
            "language":row["language"],
            "id":row["id"],
            "text":row["text"],
            "gold_intent":row["gold_intent"],
            "pred_intent":str(obj.get("intent","")),
            "intent_correct":normalize_text(obj.get("intent","")) == normalize_text(row["gold_intent"]),
            "raw_output":raw,
        }
        fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
        fout.flush()

print("✅ MTOP evaluation complete.")

## Optional MLCL extension

MLCL is useful but is not required for this notebook to strengthen the paper.
MASSIVE already supplies independent natural multilingual intent + slot evidence,
and MTOP supplies an independent multilingual intent benchmark.

If you have the MLCL artifact ZIP, run the next cells. Otherwise skip them.

In [ ]:
# CELL 13 — OPTIONAL MLCL ZIP UPLOAD

MLCL_ROOT = CACHE / "mlcl"
MLCL_AVAILABLE = False

try:
    from google.colab import files
    print("Optional: upload the MLCL repository/data ZIP now.")
    print("Cancel the upload dialog if you want to skip MLCL.")
    uploaded = files.upload()

    zips = [name for name in uploaded if name.lower().endswith(".zip")]
    if zips:
        zname = zips[0]
        zp = CACHE / zname
        zp.write_bytes(uploaded[zname])

        if MLCL_ROOT.exists():
            shutil.rmtree(MLCL_ROOT)
        MLCL_ROOT.mkdir(parents=True)

        with zipfile.ZipFile(zp) as z:
            for info in z.infolist():
                target=(MLCL_ROOT/info.filename).resolve()
                if not str(target).startswith(str(MLCL_ROOT.resolve())):
                    raise RuntimeError("Unsafe path in MLCL ZIP")
            z.extractall(MLCL_ROOT)

        MLCL_AVAILABLE=True
        print("✅ MLCL extracted.")
    else:
        print("MLCL skipped.")
except Exception as e:
    print("MLCL upload skipped:", repr(e))

print("MLCL_AVAILABLE =", MLCL_AVAILABLE)

In [ ]:
# CELL 13B — OPTIONAL MLCL SCHEMA DISCOVERY
#
# This cell never blocks the core MASSIVE/MTOP experiment.

def read_any_table(path: Path):
    try:
        s = path.suffix.lower()
        if s == ".jsonl":
            return pd.read_json(path, lines=True)
        if s == ".json":
            obj = json.loads(path.read_text(encoding="utf-8"))
            if isinstance(obj, list):
                return pd.json_normalize(obj)
            if isinstance(obj, dict):
                for key in ["data","examples","items","records"]:
                    if isinstance(obj.get(key), list):
                        return pd.json_normalize(obj[key])
                return pd.json_normalize([obj])
        if s == ".csv":
            return pd.read_csv(path)
        if s in {".parquet",".pq"}:
            return pd.read_parquet(path)
    except Exception:
        return None
    return None

def first_present(mapping, names, default=None):
    for name in names:
        if name in mapping and mapping[name] is not None:
            return mapping[name]
    return default

def infer_mlcl_language(path, row):
    blob = (str(path) + " " + json.dumps(row, ensure_ascii=False, default=str)).lower()
    rules = [
        ("hindi","hi"), ("hi_","hi"), ("chinese","zh"), ("zh_","zh"),
        ("igbo","ig"), ("ig_","ig"), ("english","en"), ("en_","en"),
    ]
    for token, lang in rules:
        if token in blob:
            return lang
    return "unknown"

def table_score(path, df):
    if df is None or df.empty:
        return -1
    cols = " ".join(str(c).lower() for c in df.columns)
    p = str(path).lower()
    score = 0
    for x in ["query","question","prompt","function","tool","answer","ground","target"]:
        if x in cols:
            score += 1
    for x in ["mlcl","bfcl","translated","hindi","igbo","chinese"]:
        if x in p:
            score += 1
    return score

MLCL_CANDIDATES = []

if MLCL_AVAILABLE:
    for p in MLCL_ROOT.rglob("*"):
        if p.is_file() and p.suffix.lower() in {".jsonl",".json",".csv",".parquet",".pq"}:
            df = read_any_table(p)
            if df is not None and len(df):
                MLCL_CANDIDATES.append((table_score(p,df),p,df))

    MLCL_CANDIDATES.sort(key=lambda x:x[0], reverse=True)

    inventory = []
    for score,p,df in MLCL_CANDIDATES[:30]:
        inventory.append({
            "score":score,
            "path":str(p.relative_to(MLCL_ROOT)),
            "rows":len(df),
            "columns":" | ".join(map(str,df.columns)),
        })
    inv_df = pd.DataFrame(inventory)
    inv_df.to_csv(RESULTS/"mlcl_candidate_tables.csv",index=False)
    display(inv_df.head(20))
else:
    print("MLCL skipped — core MASSIVE/MTOP experiment is unaffected.")

In [ ]:
# CELL 13C — OPTIONAL MLCL TOOL-CALL EVALUATION

def normalize_obj(x):
    if isinstance(x,str):
        try:
            return normalize_obj(json.loads(x))
        except Exception:
            return normalize_text(x)
    if isinstance(x,dict):
        return {str(k):normalize_obj(v) for k,v in sorted(x.items(), key=lambda kv:str(kv[0]))}
    if isinstance(x,list):
        return [normalize_obj(v) for v in x]
    return x

def function_and_args(obj):
    if isinstance(obj,str):
        try:
            obj=json.loads(obj)
        except Exception:
            return "",{}
    if isinstance(obj,list) and obj:
        return function_and_args(obj[0])
    if not isinstance(obj,dict):
        return "",{}

    if isinstance(obj.get("function"),dict):
        fn=str(obj["function"].get("name",""))
        args=obj["function"].get("arguments",{})
    else:
        fn=str(first_present(obj,["function","name","tool_name","function_name"],""))
        args=first_present(obj,["arguments","args","parameters","kwargs"],{})

    if isinstance(args,str):
        try:
            args=json.loads(args)
        except Exception:
            args={}
    if not isinstance(args,dict):
        args={}
    return fn,args

def discover_mlcl_records():
    records=[]
    for score,path,df in MLCL_CANDIDATES:
        if score < 2:
            continue
        for _,r in df.iterrows():
            rd=r.to_dict()
            query=first_present(rd,["question","query","user_query","prompt","input","utterance","user"])
            tools=first_present(rd,["functions","tools","function_list","tool_list","function","tool"])
            gold=first_present(rd,["answer","answers","ground_truth","ground_truths","expected",
                                   "reference","target","tool_call","tool_calls"])
            if query is None or tools is None or gold is None:
                continue
            records.append({
                "source_file":str(path.relative_to(MLCL_ROOT)),
                "language":infer_mlcl_language(path,rd),
                "query":str(query),
                "tools":tools,
                "gold":gold,
            })
    return records

MLCL_SYSTEM = """You are evaluating multilingual tool calling.
Return ONLY valid JSON:
{
  "function":"exact selected function name",
  "arguments":{"parameter":"value"}
}
Rules:
- Use the supplied tool interface exactly.
- Do not translate function names or argument keys.
- Preserve execution-relevant values from the user's request.
- Do not add explanation.
"""

mlcl_pred = pd.DataFrame()

if MLCL_AVAILABLE:
    records=discover_mlcl_records()
    print("Discovered MLCL-compatible records:",len(records))

    if records:
        rdf=pd.DataFrame(records)

        selected=[]
        for lang,g in rdf.groupby("language"):
            n=len(g) if LIMITS["mlcl_per_lang"] is None else min(LIMITS["mlcl_per_lang"],len(g))
            selected.append(g.sample(n=n,random_state=SEED))
        eval_df=pd.concat(selected,ignore_index=True)

        out_path=RAW/"mlcl_predictions.jsonl"
        done={}
        if out_path.exists():
            for line in out_path.read_text(encoding="utf-8").splitlines():
                try:
                    x=json.loads(line); done[x["row_key"]]=x
                except Exception:
                    pass

        with out_path.open("a",encoding="utf-8") as fout:
            for idx,row in tqdm(eval_df.iterrows(),total=len(eval_df),desc="MLCL"):
                key=f"{row['source_file']}::{idx}"
                if key in done:
                    continue

                user=f"""Language: {row['language']}
User request:
{row['query']}

Available tool interface:
{json.dumps(row['tools'],ensure_ascii=False,default=str)}
"""
                raw=call_llm(MLCL_SYSTEM,user,max_new_tokens=256)
                pred=extract_json(raw) or {}

                pfn,pargs=function_and_args(pred)
                gfn,gargs=function_and_args(row["gold"])

                pkeys=set(map(str,pargs.keys()))
                gkeys=set(map(str,gargs.keys()))
                shared=pkeys & gkeys
                value_scores=[
                    normalize_text(pargs[k])==normalize_text(gargs[k])
                    for k in shared
                ]

                rec={
                    "row_key":key,
                    "dataset":"MLCL",
                    "language":row["language"],
                    "query":row["query"],
                    "gold_function":gfn,
                    "pred_function":pfn,
                    "function_correct":normalize_text(pfn)==normalize_text(gfn),
                    "argument_key_exact":pkeys==gkeys,
                    "shared_argument_value_accuracy":(
                        float(np.mean(value_scores)) if value_scores
                        else (1.0 if not pkeys and not gkeys else 0.0)
                    ),
                    "full_tool_call_exact":(
                        normalize_text(pfn)==normalize_text(gfn)
                        and normalize_obj(pargs)==normalize_obj(gargs)
                    ),
                    "gold_arguments":gargs,
                    "pred_arguments":pargs,
                    "raw_output":raw,
                }
                fout.write(json.dumps(rec,ensure_ascii=False,default=str)+"\n")
                fout.flush()

        mlcl_pred=load_jsonl_df(out_path) if "load_jsonl_df" in globals() else pd.DataFrame(
            [json.loads(x) for x in out_path.read_text(encoding="utf-8").splitlines() if x.strip()]
        )
        print("MLCL evaluated rows:",len(mlcl_pred))
    else:
        print("⚠️ MLCL ZIP was present but no compatible query/tools/gold table was inferred.")
        print("See results/mlcl_candidate_tables.csv; MASSIVE/MTOP remain complete.")
else:
    print("MLCL not supplied; skipped.")

In [ ]:
# CELL 14 — LOAD RESULT FILES + COMPLETENESS CHECKS

def load_jsonl_df(path):
    if not path.exists():
        return pd.DataFrame()
    rows=[]
    for line in path.read_text(encoding="utf-8").splitlines():
        try:
            rows.append(json.loads(line))
        except Exception:
            pass
    return pd.DataFrame(rows)

massive_pred = load_jsonl_df(RAW/"massive_predictions.jsonl")
mtop_pred = load_jsonl_df(RAW/"mtop_predictions.jsonl")
mlcl_pred = load_jsonl_df(RAW/"mlcl_predictions.jsonl")

assert len(massive_pred) == len(massive_df), (
    f"MASSIVE predictions incomplete: {len(massive_pred)}/{len(massive_df)}"
)
assert len(mtop_pred) == len(mtop_df), (
    f"MTOP predictions incomplete: {len(mtop_pred)}/{len(mtop_df)}"
)

print("MASSIVE predictions:",len(massive_pred),"/",len(massive_df))
print("MTOP predictions:",len(mtop_pred),"/",len(mtop_df))
print("MLCL predictions:",len(mlcl_pred),"(optional)")
print("✅ Required external-dataset completeness checks passed.")

In [ ]:
# CELL 15 — METRICS + 95% BOOTSTRAP CIs

metric_rows=[]

def add_metric(dataset, group, metric, values):
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(float)
    mean,lo,hi = bootstrap_ci(vals)
    metric_rows.append({
        "dataset":dataset,
        "group":group,
        "metric":metric,
        "n":len(vals),
        "mean":mean,
        "ci95_low":lo,
        "ci95_high":hi,
    })

for dataset,df,metrics in [
    ("MASSIVE",massive_pred,[
        "intent_correct","slot_pair_f1","slot_label_f1","frame_exact",
        "surface_grounded","hallucinated_slot_value_rate",
    ]),
    ("MTOP",mtop_pred,["intent_correct"]),
]:
    for metric in metrics:
        add_metric(dataset,"ALL",metric,df[metric])
    for lang,g in df.groupby("language"):
        for metric in metrics:
            add_metric(dataset,lang,metric,g[metric])

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(RESULTS/"external_dataset_metrics.csv", index=False)

display(metrics_df.round(4))

if len(mlcl_pred):
    for metric in ["function_correct","argument_key_exact",
                   "shared_argument_value_accuracy","full_tool_call_exact"]:
        add_metric("MLCL","ALL",metric,mlcl_pred[metric])
    for lang,g in mlcl_pred.groupby("language"):
        for metric in ["function_correct","argument_key_exact",
                       "shared_argument_value_accuracy","full_tool_call_exact"]:
            add_metric("MLCL",lang,metric,g[metric])

    # Rewrite after optional MLCL additions.
    metrics_df=pd.DataFrame(metric_rows)
    metrics_df.to_csv(RESULTS/"external_dataset_metrics.csv",index=False)
    display(metrics_df.round(4))


In [ ]:
# CELL 16 — PAPER-READY TABLES

summary = pd.DataFrame([
    {
        "Dataset":"MASSIVE",
        "N":len(massive_pred),
        "Languages":massive_pred["language"].nunique(),
        "Intent accuracy":massive_pred["intent_correct"].mean(),
        "Slot-pair F1":massive_pred["slot_pair_f1"].mean(),
        "Exact frame":massive_pred["frame_exact"].mean(),
        "Surface-grounded outputs":massive_pred["surface_grounded"].mean(),
    },
    {
        "Dataset":"MTOP",
        "N":len(mtop_pred),
        "Languages":mtop_pred["language"].nunique(),
        "Intent accuracy":mtop_pred["intent_correct"].mean(),
        "Slot-pair F1":np.nan,
        "Exact frame":np.nan,
        "Surface-grounded outputs":np.nan,
    },
])

summary.to_csv(RESULTS/"paper_external_summary.csv", index=False)
display(summary.round(4))

(RESULTS/"paper_external_summary.tex").write_text(
    summary.to_latex(index=False, float_format=lambda x:f"{x:.4f}"),
    encoding="utf-8"
)

In [ ]:
# CELL 17 — PAPER FIGURES

# Headline metric plot
plot_df = pd.DataFrame([
    {"dataset":"MASSIVE","metric":"Intent accuracy","score":massive_pred["intent_correct"].mean()},
    {"dataset":"MASSIVE","metric":"Slot-pair F1","score":massive_pred["slot_pair_f1"].mean()},
    {"dataset":"MASSIVE","metric":"Exact frame","score":massive_pred["frame_exact"].mean()},
    {"dataset":"MTOP","metric":"Intent accuracy","score":mtop_pred["intent_correct"].mean()},
])

pivot = plot_df.pivot(index="dataset", columns="metric", values="score")
ax = pivot.plot(kind="bar", figsize=(8.5,5))
ax.set_ylim(0,1)
ax.set_ylabel("Score")
ax.set_title("Independent external multilingual evaluation")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES/"external_multilingual_summary.png", dpi=300, bbox_inches="tight")
plt.show()

# Language-wise performance
rows=[]
for dataset,df,metric in [
    ("MASSIVE",massive_pred,"frame_exact"),
    ("MTOP",mtop_pred,"intent_correct"),
]:
    for lang,g in df.groupby("language"):
        rows.append({
            "dataset":dataset,
            "language":lang,
            "metric":metric,
            "score":g[metric].mean(),
            "n":len(g),
        })

lang_df=pd.DataFrame(rows)
lang_df.to_csv(RESULTS/"language_breakdown.csv",index=False)

for dataset,g in lang_df.groupby("dataset"):
    fig,ax=plt.subplots(figsize=(7.5,4.5))
    ax.bar(g["language"],g["score"])
    ax.set_ylim(0,1)
    ax.set_ylabel("Score")
    ax.set_title(f"{dataset}: language-wise external performance")
    plt.tight_layout()
    plt.savefig(FIGURES/f"{dataset.lower()}_language_breakdown.png",dpi=300,bbox_inches="tight")
    plt.show()

In [ ]:
# CELL 18 — CLAIM GATE + ERROR AUDIT

claim_rows = [
    {
        "claim":"Independent natural multilingual NLU generalization",
        "status":"SUPPORTED",
        "evidence":f"MASSIVE n={len(massive_pred)}",
        "scope":"Intent + slot grounding; external natural utterances."
    },
    {
        "claim":"Independent multilingual intent generalization",
        "status":"SUPPORTED",
        "evidence":f"MTOP n={len(mtop_pred)}",
        "scope":"Intent classification only in this script-free MTOP mirror."
    },
    {
        "claim":"End-to-end NiyamTrace-X authorization safety on external datasets",
        "status":"NOT ESTABLISHED",
        "evidence":"MASSIVE/MTOP",
        "scope":"These datasets test semantic generalization, not the full authorization/execution stack."
    },
]

if len(mlcl_pred):
    claim_rows.insert(2,{
        "claim":"Independent multilingual tool-call generalization",
        "status":"SUPPORTED",
        "evidence":f"MLCL n={len(mlcl_pred)}",
        "scope":"Tool/function and argument realization; still not end-to-end NiyamTrace-X authorization."
    })
else:
    claim_rows.insert(2,{
        "claim":"Independent multilingual tool-call generalization",
        "status":"OPTIONAL / NOT RUN",
        "evidence":"MLCL artifact not evaluated",
        "scope":"MASSIVE and MTOP results remain valid independently."
    })

claims_df = pd.DataFrame(claim_rows)
claims_df.to_csv(RESULTS/"claim_gate.csv", index=False)
display(claims_df)

# Retain concrete errors for qualitative inspection.
massive_errors = massive_pred[
    (~massive_pred["intent_correct"]) |
    (~massive_pred["frame_exact"])
].copy()

mtop_errors = mtop_pred[
    ~mtop_pred["intent_correct"]
].copy()

massive_errors.to_csv(RESULTS/"massive_errors.csv",index=False)
mtop_errors.to_csv(RESULTS/"mtop_errors.csv",index=False)

print("MASSIVE error rows:",len(massive_errors))
print("MTOP error rows:",len(mtop_errors))

In [ ]:
# CELL 19 — PAPER-READY TEXT

massive_acc = massive_pred["intent_correct"].mean()
massive_slot = massive_pred["slot_pair_f1"].mean()
massive_frame = massive_pred["frame_exact"].mean()
massive_ground = massive_pred["surface_grounded"].mean()
mtop_acc = mtop_pred["intent_correct"].mean()

text = f"""# External dataset results

To test whether the semantic findings extend beyond the internally generated
NiyamTrace-X holdout, we evaluated the same local model backend on independently
released multilingual datasets. On MASSIVE (n={len(massive_pred)} across
{massive_pred['language'].nunique()} language conditions), intent accuracy was
{massive_acc:.3f}, slot-pair F1 was {massive_slot:.3f}, exact semantic-frame
accuracy was {massive_frame:.3f}, and {massive_ground:.3f} of predictions
preserved surface-grounded slot values. On the independent MTOP intent benchmark
(n={len(mtop_pred)} across {mtop_pred['language'].nunique()} languages), intent
accuracy was {mtop_acc:.3f}. These results provide external evidence for
multilingual semantic generalization; because MASSIVE and the script-free MTOP
mirror do not contain NiyamTrace-X authorization labels, they are not interpreted
as end-to-end authorization-safety evidence.
"""

(RESULTS/"paper_ready_results.md").write_text(text,encoding="utf-8")
print(text)

In [ ]:
# CELL 20 — REPRODUCIBILITY MANIFEST + EXPORT ZIP + DOWNLOAD

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):
            h.update(block)
    return h.hexdigest()

manifest = {
    "seed":SEED,
    "mode":MODE,
    "model":LOCAL_MODEL,
    "model_dtype":"float16",
    "massive_source":{
        "url":MASSIVE_URL,
        "release":"1.0",
        "languages":MASSIVE_LANGS,
        "rows":len(massive_pred),
    },
    "mtop_source":{
        "repo":"mteb/mtop_intent",
        "format":"script-free JSONL",
        "languages":MTOP_LANGS,
        "rows":len(mtop_pred),
        "scope":"intent classification",
    },
    "mlcl_source":{
        "status":"evaluated" if len(mlcl_pred) else "not evaluated",
        "rows":len(mlcl_pred),
    },
    "bootstrap_replicates":LIMITS["bootstrap"],
    "claim_boundary":"External semantic generalization only; not end-to-end NiyamTrace-X authorization safety.",
}
(RESULTS/"MANIFEST.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")

# Copy scientific outputs.
for src,name in [
    (RESULTS,"results"),
    (FIGURES,"figures"),
    (RAW,"raw_predictions"),
]:
    dst=EXPORT/name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src,dst)

zip_path = BASE/"NTX_EXTERNAL_DATASET_VALIDATION_V3_RESULTS.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED,allowZip64=True) as z:
    for p in EXPORT.rglob("*"):
        if p.is_file():
            z.write(p,arcname=str(p.relative_to(EXPORT)))

with zipfile.ZipFile(zip_path) as z:
    bad=z.testzip()

print("ZIP integrity:", "PASS" if bad is None else bad)
print("ZIP:", zip_path)
print("SHA256:", sha256_file(zip_path))
print("Size MiB:", round(zip_path.stat().st_size/(1024**2),2))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Automatic download unavailable:",repr(e))

## What to send back

After the notebook finishes, upload:

`NTX_EXTERNAL_DATASET_VALIDATION_V3_RESULTS.zip`

The Scientific Reports manuscript should only be updated after the exported
metrics and error files are audited.